## Tool Calling Agent
Here we can use the Python SDK to develop the simple tool calling agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
from nat.agent.sdk import ToolCallingAgent
from nat.llm.sdk import NimLLM
from nat.plugins.langchain.sdk import CodeGenerationTool
from nat.plugins.langchain.sdk import WikiSearchTool
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0,
    max_tokens=250,
    name="nim_llm",
)

# Define a list of tools
wikipedia_search_tool = WikiSearchTool(
    max_results=3,
    name="wiki_search",
)

current_time_tool = CurrentTimeTool(
    name="current_datetime",
)

generate_code_tool = CodeGenerationTool(
    programming_language="Python",
    description="Useful to generate Python code. For any questions about code generation, you must only use this tool!",
    llm=llm,
    verbose=True,
    name="code_generation",
)

agent = ToolCallingAgent(
    tools=[wikipedia_search_tool, current_time_tool, generate_code_tool],
    llm=llm,
    verbose=True,
    handle_tool_errors=True,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
await nat_workflow.prompt("Who was Djikstra?")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.1-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/User

'Edsger Wybe Dijkstra was a Dutch computer scientist, programmer, mathematician, and science essayist. He is best known for his work on the shortest path problem, which he solved in 1956, and for his development of the first compiler for the programming language ALGOL 60. Dijkstra was born in Rotterdam in 1930 and studied mathematics and physics at the University of Leiden. He worked as a programmer at the Mathematical Centre in Amsterdam and later became a professor at the Technische Hogeschool Eindhoven. He joined Burroughs Corporation as a research fellow in 1973 and worked there until his retirement in 1999. Dijkstra received the Turing Award in 1972 for his contributions to the development of structured programming languages. He died in 2002.'

In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  wiki_search:
    _type: wiki_search
    max_results: 3
  current_datetime:
    _type: current_datetime
  code_generation:
    _type: code_generation
    llm_name: nim_llm
    verbose: true
    programming_language: Python
    description: |-
      Useful to generate Python code. For any questions about code generation, you must only use this
      tool!

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 250
    temperature: 0.0

workflow:
  _type: tool_calling_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - wiki_search
  - current_datetime
  - code_generation
  handle_tool_errors: true



In [ ]:
from pathlib import Path

from nat.eval.sdk import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

path_to_dataset = Path(os.path.curdir, "../../../../", "examples/agents/data/wikipedia.json").resolve()

accuracy_evaluator = RagasEvaluator(
    llm=llm,
    metric="AnswerAccuracy",
    name="accuracy"
)

relevance_evaluator = RagasEvaluator(
    llm=llm,
    metric="ContextRelevance",
    name="relevance"
)

response_groundedness_evaluator = RagasEvaluator(
    llm=llm,
    metric="ResponseGroundedness",
    name="groundedness"
)

evaluation = NatEvaluation(
    output_dir=Path(".tmp/nat/examples/tool_calling_agent/"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
    evaluators=[accuracy_evaluator, relevance_evaluator, response_groundedness_evaluator],
)

nat_workflow.add_evaluator(evaluation)

In [ ]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  wiki_search:
    _type: wiki_search
    max_results: 3
  current_datetime:
    _type: current_datetime
  code_generation:
    _type: code_generation
    llm_name: nim_llm
    verbose: true
    programming_language: Python
    description: |-
      Useful to generate Python code. For any questions about code generation, you must only use this
      tool!

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 250
    temperature: 0.0

workflow:
  _type: tool_calling_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - wiki_search
  - current_datetime
  - code_generation
  handle_tool_errors: true

eval:
  general:
    max_concurrency: 8
    workflow_alias: null
    output_dir: .tmp/nat/examples/tool_calling_agent
    output: null
    dataset:
      _type: json
      id_key: id
      structure:
        disable: false
        question_key: question
        answer_key: answer
        generated_answer_key: generated_answer
        tr

In [ ]:
await nat_workflow.evaluate()

Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.1-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.1-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
Evaluating Ragas nv_accuracy:   0%|          | 0/3 [00:00<?, ?it/s]



Evaluating Ragas nv_response_groundedness: 100%|██████████| 3/3 [00:00<00:00